# Microsoft Fabric Workspace access management automation

## Introduction

This tooling provides an mechanism to automatically remediate which users, groups and machine identities have access each Fabric workspaces, ensuring alignment with a desired security / RBAC configuration stored in an external configuration file.

It uses the Semantic Link Labs python package (which are Python wrappers for Fabric APIs), and depends on Utils notebooks which provide function abstraction.

### Problem statement

Access to Workspaces is managed via authorised security groups, and developers are typically granted the `Member` role is used when provisioning developers with access to Workspaces. 

However, `Members` can also [add other entities to Workspaces](https://learn.microsoft.com/en-us/power-bi/collaborate-share/service-roles-new-workspaces#workspace-roles), including groups, machine identities or users, at equal or lower levels (_Member, Contributor or Viewer)_, which would violate the access control authorisation & approval policies. As the ability of `Members` to add entities cannot be disabled, it is necessary to enforce this policy via a control. 

To eliminate manual intervention required to implement this control, an automated solution was sought. 

### Solution

Fabric Capacities allow the execution of arbitrary Python / PySpark through Fabric Notebooks. This functionality has been combined with the [Semantic Link Labs Python package](https://semantic-link-labs.readthedocs.io/en/stable/sempy_labs.html) (_a Python wrapper for Fabric APIs_) to extract, evaluate and rectify [Workspace permissions](https://learn.microsoft.com/en-us/power-bi/collaborate-share/service-roles-new-workspaces#workspace-roles) for Fabric Workspaces. 

By using a desired state configuration file, the tooling only performs access enforcement on listed Workspaces, rather than deprovisioning & re-provisioning across the entire tenant, allowing for a progressive roll-out

### Architecture Diagram

![Architecture-diagram](./notebooks/01-utils/fabric-workspace-membership-validation-util.svg)

### Notes

1. _"Why aren't you just using the `Contributor` role?"_ Given the `Contributor` role in Fabric Workspaces has [restricted ability to create, manage and share Fabric assets](https://learn.microsoft.com/en-us/power-bi/collaborate-share/service-roles-new-workspaces#workspace-roles), and requires post-workspace creation configuration it is not widely used.

2. For the purposes of this challenge, I've removed the Service Principal components, as Service Principal based authentication is not supported for the Admin APIS [`labs.admin.add_user_to_workspace()`](https://learn.microsoft.com/en-au/rest/api/power-bi/admin/groups-add-user-as-admin) and [`labs.admin.delete_user_from_workspace()`](https://learn.microsoft.com/en-au/rest/api/power-bi/admin/groups-delete-user-as-admin)


In [ ]:
%pip install semantic-link-labs

In [ ]:
# Library installs
import notebookutils 
import sempy.fabric as fabric
import sempy_labs as labs
import sempy_labs.admin as admin
import pandas as pd
import numpy as np

In [ ]:
# get current workspace name - needed for Managed Private Endpoint set up
workspace_name = fabric.resolve_workspace_name()

# Instructions

Normally this module would be split across many smaller notebooks, and there would be an "installer" notebook to get the files all loaded up into your workspaces with neat folders. For the purposes of the 2026 Fabric Semantic Link Developer Experience Challenge, the multi-notebook module has been consolidated down to one file, and runs all as a user - no SP contexts :(

If you would like the neat multi-part solution, with abstracted functions, service principal contexts, auto-installation from Github, and solution architecture digrams, refer to [Argel-Tal/fabric-configuration-management](https://github.com/Argel-Tal/fabric-configuration-management).

A live demo & presentation on this solution can be found [here](https://youtu.be/yZ9YKUXDFf4?t=3329)

## Automatic bits
### INSTRUCTIONS: Placeholder variables

**READ THIS**

In the below code you will find placeholder values indicated by `[...]`, you will need to replace these (including the `[` `]` symbols) with values from your own tenant.

#### Managed Private Endpoints

These allow your workspace to communicate out to Private Network protected resources. You will need to establish these in all your Workspaces _(dev, uat, stg, prd, ...)_


The below code will request that resources be created for the current workspace, and the Azure Resource owner will need to approve the MPE's under the resource's networking menu. 

For more info see [Microsoft's documentation](https://learn.microsoft.com/en-au/fabric/security/security-managed-private-endpoints-create)

In [ ]:
## REPLACE THE PLACEHOLDER VALUES <...>

# create the storage container MPE
labs.managed_private_endpoint.create_managed_private_endpoint(
    (workspace_name + "-adlg2"), 
    [ADLG2 container resource ID], # TODO:replace this
    "dfs",
    ("Managed Private Endpoint for Fabric workspace: " + workspace_name)
)

# create the keyvault MPE
labs.managed_private_endpoint.create_managed_private_endpoint(
    (workspace_name + "-kv"), 
    [Key Vault resource ID], # TODO: replace this
    "vault",
    ("Managed Private Endpoint for Fabric workspace: " + workspace_name)
)

## Manual bits
### Environment

These notebooks rely on the [Semantic Link Labs package](https://semantic-link-labs.readthedocs.io/en/stable/index.html) ([Github](https://github.com/microsoft/semantic-link-labs/tree/main)). This notebook uses `%pip install` to load it, but when running in Pipelines and such you will need to make a Workspace Environment. Currently SLL does not support adding packages to an Environment (the API does). 

Environments are created as workspace artifacts, and then attached to each notebook using metadata, or the bar along the top in the notebook editor.

**Required libraries:**

Library | Version
--------|--------
Fabric  | 3.2.2
semantic-link-labs | 0.13.

### Variables

In this module, there is an assumed Azure Data Lake Gen2 enabled Azure Storage Account. 

Within this Account, it assumes a Container holding configuration data, and also another container with a subset of workspaces & domains used for testing. 

The two containers enables you to run these modules against a small list of workspaces _(files in the dev subdirectory)_, refine these modules, add to them, ... while running them in a limited blast-radius mode. Then when you're ready, you can push your content to a Production workspace, update the active variable set using a Fabric Deployment Pipeline, and all runs in the Production Workspace will automatically reference the longer list, and affect those Workspaces.

I used the names: 
- Production: `fabric-management`
- Testing: `dev-fabric-management`

Other useful variables are also stored in a variable library as follows:


name | description | value type | variable value  | Alternate value _(`Dev` workspace, if blank use default)_
:---:|-------------|------------|-----------------|-------
tenant_id | ID of your tenancy | GUID | [YOUR TENANT ID GOES HERE] |
common_capacity | ID of main Fabric capacity (assumes there's a default you assign to if assigning) | GUID | [YOUR PRIMARY FABRIC CAPACITY ID GOES HERE] |
fabric_admins | security group containing fabric admins | String | [YOUR FABRIC ADMINS GROUP ID GOES HERE] |
admin_storage_account | ADLG2 Storage Account name, which contains the configuration files | String | [YOUR ADMIN STORAGE ACCOUNT NAME GOES HERE] | 
admin_filepath | path inside the container where configuration files are stored | String | `fabric-management` | `dev-fabric-management`
admin_sp_app_id | App Reg ID of Service principal which can call Fabric Admin APIs" | GUID | [YOUR ADMIN SERVICE PRINCIPAL APP ID GOES HERE] |
keyvault_uri | URI of the Key Vault | String | `https://[YOUR KEY VAULT NAME].vault.azure.net/` | 
orchestration_sp_app_id | App Reg ID of Service principal which has access to data and is used to run workspace content | GUID | [YOUR ORCHESTRATION SERVICE PRINCIPAL APP ID GOES HERE]
orchestration_sp_app_object_id | Object ID of Service principal which has access to data and is used to run workspace content - needed for assignments | GUID | [YOUR ORCHESTRATION SERVICE PRINCIPAL OBJECT ID GOES HERE]


## Set up check-point
**To make these snippets work, you will need to have set up the Container, the service principals, the keyvaults, the tenant settings, and the variable library.**

Again, if you want a neat set up for this with different components isolated into different notebooks for maintainability, refer to [Argel-Tal/fabric-configuration-management](https://github.com/Argel-Tal/fabric-configuration-management). This specific file you are reading now has been made for the 2026 SLL challenge, and will not be revisited with future enhancements!  

**Stuff you will need:**

Application | Entity | Notes
:----------:|--------|---------
Azure DevOps | Configuration file `.csv` | master copy
Azure Storage Account | Configuration file `.csv` | Created / Updated by DevOps pipeline
EntraID | Security group for Service Principals allowed to call EDIT Fabric Admin APIs | 
EntraID| Service Principal with access to storage containers | Connects to the storage container
Azure Keyvault | Service Principal secrets  | Accessed via Keyvault queries within Fabric Notebooks
Microsoft Fabric | Fabric Capacity | 
Microsoft Fabric | Workspace(s) bound to Fabric Capacity | Also connected to Git & Deployment pipelines for CI/CD

**Stuff that is nice to have:**

Application | Entity | Notes
:----------:|--------|---------
Azure DevOps    | Pipeline | **Trigger:** Changes to configuration file subdirectory on Main branch _(via Pull Request Completion)_<br>**Action:** upload (override) Configuration files into an Azure Storage Account<br>**Action:** trigger notebook execution
Azure Storage Account | Approved Private Endpoint | Created by the above SLL snippet (fill in the `[...]` placeholders) - needed if you have public access disabled for this resource
Azure Keyvault  | Approved Private Endpoint | Created by the above SLL snippet (fill in the `[...]` placeholders) - needed if you have public access disabled for this resource

# The actual code

## Configuration bits & pieces
This section is common to Workspace management, domain management, and other notebooks I run as the Fabric Admin

In [ ]:
# get the env variables
env_vars = notebookutils.variableLibrary.getLibrary("fabric-management-vars") # again, this won't work if you haven't set up the variable library in your workspaceeeeee

In [ ]:
# environment params
common_capacity = env_vars.getVariable('common_capacity')
fabric_admins = env_vars.getVariable('fabric_admins')

# set up to mount containers
storage_account_name = env_vars.getVariable('admin_storage_account')
container_name = env_vars.getVariable('admin_filepath')
mount_path = "/" + storage_account_name
lakehouse_path = "abfss://" + container_name + "@" + storage_account_name + ".dfs.core.windows.net"
data_types = ['Files', 'Tables']

# dynamic vars
active_workspace_name = fabric.resolve_workspace_name()
active_workspace_id = fabric.get_workspace_id()

In [ ]:
# Mount external datalake "admin_lake" which contains your desired configuration files into specified local path
mssparkutils.fs.mount(
    lakehouse_path,
    mount_path
)
# check there's stuff in there
mssparkutils.fs.ls(f"file://{mssparkutils.fs.getMountPath(mount_path)}")

In [ ]:
## Need to check that user has Fabric Admin perms active when running notebook interactively when working through Workspace permissions management, as Admin versions of AddUserToWorkspace() and DeleteUserFromWorkspace() APIs do not currently support SP contexts
try:
    df = admin.list_capacities()
    print("User Authorisation successful")
except: 
    notebook_config_auth_failure_dict = {
            "message": "Authorisation failed: PBI API call failed as User, refer to Error code"
        }   
    mssparkutils.notebook.exit(notebook_config_auth_failure_dict)

## Set up, functions and more

In [ ]:
# asserted column names, as the shortcut from Azure Data lake doesnt support underscores `_` in colnames, but the APIs return with spaces ` `
# this assertion normalises the `target state` and API return `current state` tables
workspace_access_column_names = ["workspace", "Email Address", "Group User Access Right", "Identifier", "Graph Id", "Principal Type"]
workspace_access_filename = 'workspace-membership-intended.csv'
path = f"{lakehouse_path}/{workspace_access_filename}"

In [ ]:
# Load data into pandas DataFrame
workspace_members_spark = spark.read.format("csv").option('header', 'true').load(path)
workspace_membership_intended = workspace_members_spark.toPandas()
# data cleaning
workspace_membership_intended = workspace_membership_intended.replace(np.nan, None)
workspace_membership_intended.drop('User_Name', axis=1, inplace=True) # drop this so if the group is renamed we don't have to worry about it
workspace_membership_intended.columns = workspace_access_column_names

# get a unique list of workspaces to iterate over
workspaces = set(workspace_membership_intended['workspace'])

# review what you'll be reviewing & modifying permissions on
display(workspace_membership_intended)

In [ ]:
def enforce_workspace_membership(workspace_id):
    """
    Compares the current state of workspace membership with the target state (both in pandas data-frame format) and performs necessary updates (adding or removing users/groups) to align the current state with the target state.
    Requires active Fabric administrator rights, as labs.admin.delete_user_from_workspace / labs.admin.add_user_to_workspace don't currently support SP based auth (Github issue: https://github.com/microsoft/semantic-link-labs/issues/1104)

    Parameters
    ----------
    - workspace_id: unique ID for the workspace, available in the URL, or via plugins / APIs

    Returns
    -------
    None
    This function does not return any value. It adds & removes users/groups/service principals from the workspace based on the differences between `current_state_df` and `target_state_df`.

    Notes
    -----
    - If there are no differences between the current state and the target state, no operations are performed.
    - If differences exist:
    - Users/groups to be added are identified and processed.
    - Users/groups to be removed are identified and processed.
    - For removing groups, a flag is passed to distinguish them from service principals, as otherwise the underlying Fabric API throws an error.

    - Is sensitive to group name changes

    Internally generates
        
    - current_state_df : pandas.DataFrame
        A DataFrame representing the current state of workspace membership. It is generated by the Fabric Semantic-Link Labs Admin package function `list_workspace_users()` 
    - target_state_df : pandas.DataFrame
        A DataFrame representing the desired target state of workspace membership. It' structure should match that which is returned by Semantic-Link Labs Admin package function `list_workspace_users()`. This is what the workspace access policies will be updated to match.


    Examples
    --------
    >>> import pandas as pd
    >>> current_state_df = pd.DataFrame({
        'Identifier': ['user1', 'user2', 'group1'],
        Principal Type': ['User', 'User', 'Group'],
        Group User Access Right': ['Admin', 'Member', 'Contributor'],
        workspace': ['Workspace1', 'Workspace1', 'Workspace1']
        })
    >>> target_state_df = pd.DataFrame({
        'Identifier': ['user1', 'user3', 'group1'],
        'Principal Type': ['User', 'User', 'Group'],
        'Group User Access Right': ['Admin', 'Member', 'Contributor'],
        'workspace': ['Workspace1', 'Workspace1', 'Workspace1']
        })
    >>> enforce_workspace_membership(current_state_df, target_state_df)
    some differences
    start removing users and principals
    removed `user2`
    removed all users and principals
    removing groups
    removed all groups
    end remove logic
    start adding users, groups and principals
    added `user3`
    end add logic

    # Description generated partially by QChat
    """

    # get desired state
    target_state_df = workspace_membership_intended[workspace_membership_intended['workspace'] == workspace_id]

    # get current state - with error handling (requires Fabric Admin)
    ## check the workspace exists
    ### This API function returns a 0 row dataframe if not found (not an error), so can safely call it outside the try-catch
    workspace_details = labs.admin.list_workspaces(workspace = workspace_id)
    try: 
        current_state_df = labs.admin.list_workspace_users(workspace_id)
    except: 
        manage_workspace_members_invalid_dict = {
            "message": "Target workspace is does not exist",
            "invalid_workspace": workspace_id
        }
        # early termination if we can't find the workspace
        mssparkutils.notebook.exit(manage_workspace_members_invalid_dict)
    
    ## now we know it exits, has it been deleted? If so, halt and prompt to restore / remove from list 
    if workspace_details['State'].iloc[0] == 'Deleted':
        manage_workspace_members_invalid_dict = {
                "message": "Target workspace has been deleted, remove it from the mapping, or restore it",
                "invalid_workspace": workspace_id
        }   
        mssparkutils.notebook.exit(manage_workspace_members_invalid_dict)

    ## Okay cool, now we know it exists and hasn't been deleted!!

    # normal the structure between API return and target state df
    current_state_df.insert(0, 'workspace', workspace_id)
    
    # drop the user name column in-case this throws issues - esp w/ group renames
    current_state_df.drop('User Name', axis=1, inplace=True)
    current_state_df.sort_values(workspace_access_column_names, inplace= True)
    target_state_df = target_state_df.sort_values(by = workspace_access_column_names)

    '''
    ------------------- READ THIS COMMENT --------------------------
    
    Because there is currently no Service Principal support for the Admin APIS `labs.admin.add_user_to_workspace()` and `labs.admin.delete_user_from_workspace()`, we are forced to do this component outside of SP contexts.
    Have raised a feature enhancement request on SLL github: https://github.com/microsoft/semantic-link-labs/issues/1104

    If the SP context for AddUser / DeleteUser (from Workspace) Admin APIs is added, we can then indent this entire function under 1x `with labs.service_principal_authentication(...)` block
    
    -------------- THANK YOU FOR YOUR ATTENTION --------------------

    '''

    print("Checking Workspace: " + target_state_df['workspace'].iloc[0])
    ## Check if there's variation between the target state and the actual state
    if target_state_df.reset_index(drop=True).equals(current_state_df.reset_index(drop=True)):
        print("No differences")
    else:
        print("some differences")
        # generate delta between desired and actual
        delta_df = pd.merge(target_state_df, current_state_df, on=['Identifier'], how='outer',indicator=True)
        # split the adds and removes into their own little DFs
        to_remove = delta_df[delta_df['_merge'] == "right_only"]
        to_add = delta_df[delta_df['_merge'] == "left_only"]
        # Identify what permissions need to be updated
        both = delta_df[delta_df['_merge'] == "both"]
        to_update_within = both[both['Group User Access Right_x'] != both['Group User Access Right_y']]

        # get the list of users to add to the workspace in a neat 7 tidy format
        to_add = to_add.iloc[:, 0:6]
        to_add.columns = workspace_access_column_names
        # get the list of users to remove from the workspace in a neat 7 tidy format
        temp = to_remove.iloc[:,6:11]
        temp.insert(4, 'Identifier', to_remove['Identifier'])
        to_remove = temp
        to_remove.columns = workspace_access_column_names

        ## start processing changes
        # do we need to modify anyone's perms?
        if pd.DataFrame(to_update_within).size == 0: 
            print("no one to adjust")
        else:
            print('start modifying users, groups and principals')
            for index, row in to_update_within.iterrows():
                # need to check what type of person we're adding, as needs to be email for ppl and Entra Object ID for users
                if row['Principal Type_x'] == "User":
                    labs.admin.add_user_to_workspace(
                    row['Email Address_x'], # Email address is alias for UPN, Graph Object ID throws errors
                    row['Group User Access Right_x'],
                    row['Principal Type_x'],
                    row['workspace_x']
                    )
                else:
                    labs.admin.add_user_to_workspace(
                    row['Identifier'], # Graph Object ID throws for SPs and Groups
                    row['Group User Access Right_x'],
                    row['Principal Type_x'],
                    row['workspace_x']
                    )
            print('end modification logic')

        # do we need to remove anyone? 
        if pd.DataFrame(to_remove).size == 0:
            print("no removals needed")
        else:
            # logic to go here
            users_sp_to_rm = to_remove[to_remove['Principal Type'] != "Group"]
            groups_to_rm = to_remove[to_remove['Principal Type'] == "Group"]
            print("start removing users and principals")
            for index, row in users_sp_to_rm.iterrows():
                labs.admin.delete_user_from_workspace(
                row['Identifier'], 
                row['workspace']
                )
            print("removed all users and principals")
            print("removing groups")
            for index, row in groups_to_rm.iterrows():
                labs.admin.delete_user_from_workspace(
                row['Identifier'],
                row['workspace']
                # have to add a flag when removing groups, otherwise it think's it's the ID of an SP: # https://learn.microsoft.com/en-au/rest/api/power-bi/admin/groups-delete-user-as-admin
                , True   
                )
            print ("removed all groups")
        # do we need to add anyone?
        if pd.DataFrame(to_add).size == 0: 
            print("no one to add")
        else:
            print('start adding users, groups and principals')
            for index, row in to_add.iterrows():
                # need to check what type of person we're adding, as needs to be email for ppl and Entra Object ID for users
                if row['Principal Type'] == "User":
                    labs.admin.add_user_to_workspace(
                    user = row['Email Address'], # Email address is alias for UPN, Graph Object ID throws errors
                    role = row['Group User Access Right'],
                    principal_type  = row['Principal Type'],
                    workspace  = row['workspace']
                    )
                else:
                    labs.admin.add_user_to_workspace(
                    user = row['Identifier'], # Graph Object ID for SPs and Groups
                    role = row['Group User Access Right'],
                    principal_type = row['Principal Type'],
                    workspace = row['workspace']
                    )
            print("done adding users, groups and principals")
    print('finished with workspace: '+ target_state_df['workspace'].iloc[0])

## Start doing stuff!

In [ ]:
# this will loop through all workspaces in the managed workspace list (see lakehouse), and enforce the desired access configuration - eliminating any permission drift
for workspace_id in workspaces:
    enforce_workspace_membership(workspace_id)

In [ ]:
notebookutils.notebook.exit("Success")